In [1]:
# import torchmetrics
import torcheval
import torch
import torch_xla as xla
import torch_xla.core.xla_model as xm
import os
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms
import torchvision
from tqdm import tqdm

In [2]:
from torcheval.metrics.image.fid import FrechetInceptionDistance

In [3]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [4]:
class MyDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        self.image_file_name = os.listdir(image_paths)

        
    def __getitem__(self, index):
        # image_path = self.image_paths[index]
        image_path = os.path.join(self.image_paths, self.image_file_name[index])
        x = Image.open(image_path)
        if self.transform is not None:
            x = self.transform(x)
        return x
    
    def __len__(self):
        return len(self.image_file_name)

In [5]:
fid_score = FrechetInceptionDistance(device=xm.xla_device())

In [6]:
real_image_path = "/home/djfelrl11/data/train/real_images"
fake_image_path = "/home/djfelrl11/geodiffusion/work_dirs/missing_person_512x512_240629/checkpoint/iter_2820/output_img/discriminator_guidance_on"

real_image_dataset = MyDataset(real_image_path, transform=transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
]))

fake_image_dataset = MyDataset(fake_image_path, transform=transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
]))

real_image_loader = DataLoader(real_image_dataset, batch_size=32, shuffle=False, num_workers=4, drop_last=False)
fake_image_loader = DataLoader(fake_image_dataset, batch_size=32, shuffle=False, num_workers=4, drop_last=False)

In [7]:
for real_img in tqdm(real_image_loader):
    real_img = real_img.to(xm.xla_device())
    fid_score.update(real_img, is_real=True)
    xm.mark_step()

for fake_img in tqdm(fake_image_loader):
    fake_img = fake_img.to(xm.xla_device())
    fid_score.update(fake_img, is_real=False)
    xm.mark_step()
    

100%|██████████| 94/94 [00:16<00:00,  5.74it/s]


In [8]:
print(fid_score.num_real_images)
print(fid_score.num_fake_images)

value = fid_score.compute()
print(value)

tensor(3000, device='xla:0', dtype=torch.int32)
tensor(3000, device='xla:0', dtype=torch.int32)
